# **Лабораторна робота №1**
## з дисципліни **"Обробка надвеликих масивів даних"**

##### *За темою:* **"Розподілена обробка даних в Apache Hadoop та Apache Hive"**

<div style="display: flex;">

<div style="flex: 1;">
    <i>Виконав:</i>
    <br><br>
    <i><b>Гавлицький Іван КІ-61мп</b></i>
</div>

<div style="flex: 1;">
    <i>Прийняв:</i>
    <br><br>
    <i><b>Пишнограєв<br>Іван Олександрович</b></i>
</div>

</div>

## Підготовка

In [1]:
import shutil
import sys
import urllib.request
import zipfile
from pathlib import Path
from statistics import median

import pandas as pd

from lab001.cluster import EXAMPLES_JAR, compose_up, hadoop, hdfs, run
from lab001.hive import HiveClient
from lab001.postgres import PostgresClient

COMPOSE_FILE = Path("infra/docker-compose.yml")
assert COMPOSE_FILE.exists(), "run the notebook from labs/001-hadoop-and-hive"
# Bind-mounted into namenode; must exist before `compose up`, or Docker creates it owned by root.
Path("data").mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)


def compare(queries: dict[str, str]) -> pd.DataFrame:
    rows = {}
    for label, query in queries.items():
        times = hive.benchmark(query)
        rows[label] = {"median, s": round(median(times), 2), "HDFS read, MB": round(hive.last_hdfs_read / 2**20, 1)}
    return pd.DataFrame(rows).T

## Запуск кластера

Hadoop 3.1.3 (HDFS + YARN), Hive 3.1.3 та PostgreSQL 17 у Docker (`infra/`).

In [2]:
compose_up(COMPOSE_FILE)
run(["docker", "compose", "-f", str(COMPOSE_FILE), "ps", "--format", "table {{.Name}}\t{{.Status}}"]);

$ docker compose --progress quiet -f infra/docker-compose.yml up -d --wait --wait-timeout 300
$ docker compose -f infra/docker-compose.yml ps --format 'table {{.Name}}	{{.Status}}'
NAME              STATUS
datanode          Up 32 seconds (healthy)
historyserver     Up 30 seconds (healthy)
hive-server       Up 30 seconds
namenode          Up 30 seconds (healthy)
nodemanager       Up 31 seconds (healthy)
postgres          Up 31 seconds (healthy)
resourcemanager   Up 31 seconds (healthy)


## Завдання 1. Підготовка та завантаження даних (Python & HDFS)

### Завантаження даних

Скрипт `scripts/download_data.py` завантажує вивантаження ЄДР від 08.04.2022 та конвертує XML у `data/UO.csv` і `data/FOP.csv`.

In [3]:
run([sys.executable, "scripts/download_data.py"]);

$ /home/nuinashco/Projects/iasa-bigdata/.venv/bin/python scripts/download_data.py
[download] https://web.archive.org/web/20220413051017id_/https://nais.gov.ua/files/general/2022/04/08/20220408144948-25.zip
[download] saved /home/nuinashco/Projects/iasa-bigdata/labs/001-hadoop-and-hive/data/raw/edr_2022-04-08.zip (461 MB)
[convert] UO.csv: 1,853,363 rows in 75s
[convert] FOP.csv: 5,792,593 rows in 107s


### Робота з HDFS CLI

Папка `data/` змонтована в контейнер `namenode` як `/data`. Спершу чекаємо, поки HDFS вийде з безпечного режиму (safe mode).

In [4]:
hdfs("dfsadmin", "-safemode", "wait");

$ hdfs dfsadmin -safemode wait
Safe mode is OFF


In [5]:
hdfs("dfs", "-mkdir", "-p", "/tables_data/UO", "/tables_data/FOP");

$ hdfs dfs -mkdir -p /tables_data/UO /tables_data/FOP


In [6]:
hdfs("dfs", "-put", "-f", "/data/UO.csv", "/tables_data/UO/")
hdfs("dfs", "-put", "-f", "/data/FOP.csv", "/tables_data/FOP/");

$ hdfs dfs -put -f /data/UO.csv /tables_data/UO/


$ hdfs dfs -put -f /data/FOP.csv /tables_data/FOP/


In [7]:
hdfs("dfs", "-ls", "-h", "/tables_data/UO", "/tables_data/FOP");

$ hdfs dfs -ls -h /tables_data/UO /tables_data/FOP
Found 1 items
-rw-r--r--   1 root supergroup      1.1 G 2026-10-07 21:53 /tables_data/UO/UO.csv
Found 1 items
-rw-r--r--   1 root supergroup      1.9 G 2026-10-07 21:53 /tables_data/FOP/FOP.csv


Файл розбито на блоки по 128 МБ (UO.csv ≈ 1,2 ГБ → 9 блоків), фактор реплікації — 1:

In [8]:
hdfs("fsck", "/tables_data/UO/UO.csv", "-files", "-blocks");

$ hdfs fsck /tables_data/UO/UO.csv -files -blocks
Connecting to namenode via http://namenode:9870/fsck?ugi=root&files=1&blocks=1&path=%2Ftables_data%2FUO%2FUO.csv
FSCK started by root (auth:SIMPLE) from /172.18.0.7 for path /tables_data/UO/UO.csv at Wed Oct 07 21:53:14 UTC 2026
/tables_data/UO/UO.csv 1200075063 bytes, replicated: replication=1, 9 block(s):  OK
0. BP-482417747-172.18.0.7-1791409663714:blk_1073741830_1006 len=134217728 Live_repl=1
1. BP-482417747-172.18.0.7-1791409663714:blk_1073741831_1007 len=134217728 Live_repl=1
2. BP-482417747-172.18.0.7-1791409663714:blk_1073741832_1008 len=134217728 Live_repl=1
3. BP-482417747-172.18.0.7-1791409663714:blk_1073741833_1009 len=134217728 Live_repl=1
4. BP-482417747-172.18.0.7-1791409663714:blk_1073741834_1010 len=134217728 Live_repl=1
5. BP-482417747-172.18.0.7-1791409663714:blk_1073741835_1011 len=134217728 Live_repl=1
6. BP-482417747-172.18.0.7-1791409663714:blk_1073741836_1012 len=134217728 Live_repl=1
7. BP-482417747-172.18.0.7-1

## Завдання 2. Створення таблиць в Apache Hive

Зовнішні таблиці `uo_table` та `fop_table` поверх папок HDFS, формат CSV розбирає `OpenCSVSerde`.

In [9]:
hive = HiveClient()

hive.sql(r"""
CREATE EXTERNAL TABLE IF NOT EXISTS uo_table (
    name     STRING,
    edrpou   STRING,
    address  STRING,
    boss     STRING,
    founders STRING,
    kved     STRING,
    stan     STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    "separatorChar" = ",",
    "quoteChar"     = "\""
)
STORED AS TEXTFILE
LOCATION '/tables_data/UO'
TBLPROPERTIES ("skip.header.line.count" = "1")
""")

[0.47s] CREATE EXTERNAL TABLE IF NOT EXISTS uo_table ( name STRING, edrpou STRING, address STRING, boss S...


In [10]:
hive.sql(r"""
CREATE EXTERNAL TABLE IF NOT EXISTS fop_table (
    fio     STRING,
    address STRING,
    kved    STRING,
    stan    STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    "separatorChar" = ",",
    "quoteChar"     = "\""
)
STORED AS TEXTFILE
LOCATION '/tables_data/FOP'
TBLPROPERTIES ("skip.header.line.count" = "1")
""")

[0.11s] CREATE EXTERNAL TABLE IF NOT EXISTS fop_table ( fio STRING, address STRING, kved STRING, stan STR...


In [11]:
hive.sql(r"""
SELECT *

FROM
    uo_table

LIMIT 10
""")

[2.57s] SELECT * FROM uo_table LIMIT 10


,name,edrpou,address,boss,founders,kved,stan
0,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДАШУКІВСЬКІ БЕНТОНІТИ""",00223941,"Україна, 19330, Черкаська обл., Лисянський р-н, село Дашуківка",БЕСАРАБ ВОЛОДИМИР МИХАЙЛОВИЧ,"АКЦІОНЕРИ, розмір частки - 13719925,00 грн.","08.12 Добування піску, гравію, глин і каоліну",зареєстровано
1,"ВІДДІЛ ОСВІТИ, ОХОРОНИ ЗДОРОВ'Я, КУЛЬТУРИ, СПОРТУ ЗОЛОТОНІСЬКОЇ РАЙОННОЇ ДЕР...",02147061,"Україна, 19702, Черкаська обл., місто Золотоноша, ВУЛИЦЯ БЛАГОВІЩЕНСЬКА, буд...",Дехтярук Інна Григорівна,,"84.12 Регулювання у сферах охорони здоров'я, освіти, культури та інших соціа...",зареєстровано
2,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ПРОМЕКС""",30904152,"Україна, 19322, , С. ПОЧАПИНЦІ, ЛИСЯНСЬКИЙ РАЙОН, ЧЕРКАСЬКА ОБЛАСТЬ",,,,припинено
3,"ВІДКРИТЕ АКЦІОНЕРНЕ ТОВАРИСТВО ""ЛИСЯНСЬКИЙ МОЛОКОЗАВОД""",01530272,"Україна, 19300, ПРОВ. БУЖАНСЬКИЙ, 32, СМТ ЛИСЯНКА, ЛИСЯНСЬКИЙ РАЙОН, ЧЕРКАСЬ...",,,,припинено
4,"КОМУНАЛЬНЕ ПІДПРИЄМСТВО ""ЛИСЯНСЬКЕ ВИРОБНИЧЕ УПРАВЛІННЯ ЖИТЛОВО-КОМУНАЛЬНОГО...",03357079,"Україна, 19300, ВУЛ.КИЇВСЬКА, 86, СМТ ЛИСЯНКА, ЛИСЯНСЬКИЙ РАЙОН, ЧЕРКАСЬКА О...",,,,припинено
5,"""ЧЕСНІВСЬКЕ""",32242050,"Україна, 19310, Черкаська обл., Лисянський р-н, село Чеснівка",ГЕРМАН ПАВЛО ІВАНОВИЧ,"ГЕРМАН ПАВЛО ІВАНОВИЧ, розмір частки - 0,00 грн.",01.11.0 ВИРОЩУВАННЯ ЗЕРНОВИХ ТА ТЕХНІЧНИХ КУЛЬТУР,припинено
6,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ "" АРБАТ - АГРО """,31100636,"Україна, 19100, ВУЛ. ВІННИЦЬКА, 27, М. МОНАСТИРИЩЕ, МОНАСТИРИЩЕНСЬКИЙ РАЙОН,...",,,,припинено
7,ЛИСЯНСЬКЕ ПІДПРИЄМСТВО ТЕПЛОВИХ МЕРЕЖ,14215690,"Україна, 19300, ВУЛ.ЧЕРВОНОАРМІЙСЬКА, 2, СМТ. ЛИСЯНКА, ЛИСЯНСЬКИЙ РАЙОН, ЧЕР...",ДЕРКАЧ ПЕТРО МИХАЙЛОВИЧ,"ЛИСЯНСЬКА СЕЛИЩНА РАДА, розмір частки - 0,00 грн.",40.3 ПОСТАЧАННЯ ПАРИ ТА ГАРЯЧОЇ ВОДИ,припинено
8,МАР'ЯНІВСЬКИЙ ЗАКЛАД ЗАГАЛЬНОЇ СЕРЕДНЬОЇ ОСВІТИ,33335512,"Україна, 19344, Черкаська обл., Лисянський р-н, селище Мар'янівка, ВУЛИЦЯ ШК...",ГАВРИЛЮК ІРИНА МИКОЛАЇВНА,ВИНОГРАДСЬКА СІЛЬСЬКА РАДА,85.31 Загальна середня освіта,зареєстровано
9,МАНЬКІВСЬКА ДИТЯЧА ШКОЛА МИСТЕЦТВ,24355966,"Україна, 20101, Черкаська обл., Уманський р-н, селище міського типу Маньківк...",БАС СТАНІСЛАВ АНАТОЛІЙОВИЧ,МАНЬКІВСЬКА СЕЛИЩНА РАДА,85.52 Освіта у сфері культури,зареєстровано


In [12]:
hive.sql(r"""
SELECT *

FROM
    fop_table

LIMIT 10
""")

[0.24s] SELECT * FROM fop_table LIMIT 10


,fio,address,kved,stan
0,ФЕДОРЕНКО КОСТЯНТИН ВОЛОДИМИРОВИЧ,"Україна, 15300, Чернігівська обл., Корюківський р-н, місто Корюківка, вул.Па...",49.41 Вантажний автомобільний транспорт,зареєстровано
1,Рябченко Юрій Максимович,"Україна, 12601, Житомирська обл., Житомирський р-н, селище міського типу Бру...",47.11 Роздрібна торгівля в неспеціалізованих магазинах переважно продуктами ...,зареєстровано
2,ТЕЛІШЕВСЬКА ОЛЬГА БОГДАНІВНА,"Україна, 79034, Львівська обл., місто Львів, вул. Стуса В., будинок 24, квар...",69.10 Діяльність у сфері права,зареєстровано
3,ТИМОШЕНКО ДМІТРІЙ ЮРІЙОВИЧ,"Україна, 03041, місто Київ, вулиця Блакитного, будинок 8","56.10 Діяльність ресторанів, надання послуг мобільного харчування",припинено
4,ШИМКІВ ІРИНА АНАТОЛІЇВНА,"Україна, 39601, Полтавська обл., місто Кременчук, вулиця Європейська, будино...","46.33 Оптова торгівля молочними продуктами, яйцями, харчовими оліями та жирами",зареєстровано
5,Корж Єлизавета Володимирівна,"Україна, 02232, місто Київ, вул.Бальзака Оноре де, будинок 5-А, квартира 155","47.76 Роздрібна торгівля квітами, рослинами, насінням, добривами, домашніми ...",зареєстровано
6,Цуркан Анжела Борисівна,"Україна, 66502, Одеська обл., Подільський р-н, селище міського типу Любашівк...","96.09 Надання інших індивідуальних послуг, н.в.і.у.",зареєстровано
7,Думбрава Тамара Іванівна,"Україна, 65016, Одеська обл., місто Одеса, пров.Компасний, будинок 1-А, квар...",47.73 Роздрібна торгівля фармацевтичними товарами в спеціалізованих магазинах,зареєстровано
8,Шавеко Євген Ігорович,"Україна, 68421, Одеська обл., Болградський р-н, село Теплиця, вул.Верхня, бу...",47.79 Роздрібна торгівля уживаними товарами в магазинах,зареєстровано
9,Головчук Сергій Федорович,"Україна, 65000, Одеська обл., місто Одеса, вулиця Героїв Прикордонників, буд...",62.01 Комп'ютерне програмування,зареєстровано


Метадані таблиці: тип `EXTERNAL_TABLE` та розташування даних у HDFS:

In [13]:
hive.sql("DESCRIBE FORMATTED uo_table").fillna("")

[0.12s] DESCRIBE FORMATTED uo_table


,col_name,data_type,comment
0,# col_name,data_type,comment
1,name,string,from deserializer
2,edrpou,string,from deserializer
3,address,string,from deserializer
4,boss,string,from deserializer
5,founders,string,from deserializer
6,kved,string,from deserializer
7,stan,string,from deserializer
8,,,
9,# Detailed Table Information,,


## Завдання 3. Виконання аналітичних запитів (HiveQL)

### 3.1. Агрегація даних

Для порівняння ті самі CSV завантажено в PostgreSQL.

In [14]:
pg = PostgresClient()

pg.sql(r"""
CREATE TABLE IF NOT EXISTS uo (
    name     TEXT,
    edrpou   TEXT,
    address  TEXT,
    boss     TEXT,
    founders TEXT,
    kved     TEXT,
    stan     TEXT
);

CREATE TABLE IF NOT EXISTS fop (
    fio     TEXT,
    address TEXT,
    kved    TEXT,
    stan    TEXT
);

TRUNCATE uo, fop;

COPY uo FROM '/data/UO.csv' WITH (FORMAT csv, HEADER true);
COPY fop FROM '/data/FOP.csv' WITH (FORMAT csv, HEADER true);

VACUUM ANALYZE uo;
VACUUM ANALYZE fop;
""")

[0.01s] CREATE TABLE IF NOT EXISTS uo ( name TEXT, edrpou TEXT, address TEXT, boss TEXT, founders TEXT, k...
[0.00s] CREATE TABLE IF NOT EXISTS fop ( fio TEXT, address TEXT, kved TEXT, stan TEXT )
[0.00s] TRUNCATE uo, fop


[19.64s] COPY uo FROM '/data/UO.csv' WITH (FORMAT csv, HEADER true)


[33.67s] COPY fop FROM '/data/FOP.csv' WITH (FORMAT csv, HEADER true)


[2.59s] VACUUM ANALYZE uo


[2.44s] VACUUM ANALYZE fop


Hive може відповідати на `count(*)` з метаданих (статистики таблиці), не читаючи дані. Вимикаємо це, щоб кожен запит виконувався як MapReduce-задача. Кожен запит запускається 3 рази, у таблиці — медіана.

In [15]:
hive.sql("SET hive.compute.query.using.stats=false")

COUNT_QUERY = r"""
SELECT
    COUNT(*) AS rows_count

FROM
    {table}
"""

counts = pd.DataFrame(
    {
        "rows": [
            hive.sql(COUNT_QUERY.format(table="uo_table")).iloc[0, 0],
            hive.sql(COUNT_QUERY.format(table="fop_table")).iloc[0, 0],
        ],
        "Hive, s": [
            median(hive.benchmark(COUNT_QUERY.format(table="uo_table"))),
            median(hive.benchmark(COUNT_QUERY.format(table="fop_table"))),
        ],
        "PostgreSQL, s": [
            median(pg.benchmark(COUNT_QUERY.format(table="uo"))),
            median(pg.benchmark(COUNT_QUERY.format(table="fop"))),
        ],
    },
    index=["UO", "FOP"],
)
counts

[0.01s] SET hive.compute.query.using.stats=false


[30.72s, HDFS read 1.1 GB] SELECT COUNT(*) AS rows_count FROM uo_table


[36.77s, HDFS read 1.9 GB] SELECT COUNT(*) AS rows_count FROM fop_table


median 28.18s (runs: 28.18, 27.55, 29.82), HDFS read 1.1 GB | SELECT COUNT(*) AS rows_count FROM uo_table


median 37.16s (runs: 36.18, 38.31, 37.16), HDFS read 1.9 GB | SELECT COUNT(*) AS rows_count FROM fop_table


median 0.15s (runs: 0.17, 0.15, 0.14) | SELECT COUNT(*) AS rows_count FROM uo


median 0.29s (runs: 0.29, 0.28, 0.30) | SELECT COUNT(*) AS rows_count FROM fop


,rows,"Hive, s","PostgreSQL, s"
UO,1853363,28.177506,0.148209
FOP,5792593,37.161205,0.289515


### 3.2. Аналітика даних: нумерація в межах адреси

Віконна функція `row_number()` нумерує юридичних осіб у межах кожної адреси. Найбільший номер на адресі — кількість зареєстрованих на ній компаній.

In [16]:
hive.sql(r"""
CREATE OR REPLACE VIEW uo_numbered AS
SELECT
    name,
    edrpou,
    address,
    ROW_NUMBER() OVER (PARTITION BY address ORDER BY edrpou) AS rn

FROM
    uo_table
""")

[0.22s] CREATE OR REPLACE VIEW uo_numbered AS SELECT name, edrpou, address, ROW_NUMBER() OVER (PARTITION ...


In [17]:
mass_registration = hive.sql(r"""
SELECT
    address,
    MAX(rn) AS companies

FROM
    uo_numbered

GROUP BY address

ORDER BY companies DESC

LIMIT 20
""")
mass_registration

[96.00s, HDFS read 1.5 GB] SELECT address, MAX(rn) AS companies FROM uo_numbered GROUP BY address ORDER BY companies DESC LI...


,address,companies
0,"Україна, 04050, місто Київ, ВУЛИЦЯ МЕЛЬНИКОВА, будинок 12",2187
1,"Україна, 01014, місто Київ, ВУЛИЦЯ ЗВІРИНЕЦЬКА, будинок 63",1444
2,"Україна, 04119, місто Київ, ВУЛИЦЯ ЗООЛОГІЧНА, будинок 4А, офіс 139",729
3,"Україна, 03124, місто Київ, ВУЛИЦЯ МИКОЛИ ВАСИЛЕНКА, будинок 7-А",643
4,"Україна, 03127, місто Київ, ПРОСПЕКТ 40-РІЧЧЯ ЖОВТНЯ, будинок 120, корпус 1",627
5,"Україна, 02099, місто Київ, ВУЛИЦЯ ЯЛТИНСЬКА, будинок 5-Б",553
6,"Україна, 04205, місто Київ, ПРОСПЕКТ ОБОЛОНСЬКИЙ, будинок 23-А",518
7,"Україна, 01133, місто Київ, БУЛЬВАР ЛЕСІ УКРАЇНКИ, будинок 26",509
8,"Україна, 03127, місто Київ, ПРОСПЕКТ 40-РІЧЧЯ ЖОВТНЯ, будинок 100/2",508
9,"Україна, 04080, місто Київ, ВУЛИЦЯ КИРИЛІВСЬКА, будинок 82, офіс 256",502


Перші юридичні особи на адресі масової реєстрації №1:

In [18]:
top_address = mass_registration.loc[0, "address"].replace("'", "\\'")

hive.sql(rf"""
SELECT
    rn,
    edrpou,
    name

FROM
    uo_numbered

WHERE 1=1
    AND address = '{top_address}'

ORDER BY rn

LIMIT 20
""")

[61.25s, HDFS read 1.1 GB] SELECT rn, edrpou, name FROM uo_numbered WHERE 1=1 AND address = 'Україна, 04050, місто Київ, ВУЛ...


,rn,edrpou,name
0,1,13474886,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""СИНТОП"""
1,2,13668159,"ЗАКРИТЕ АКЦІОНЕРНЕ ТОВАРИСТВО ""СПЕЦ-ІНВЕСТ"""
2,3,13668780,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""КОМПАН-УКРАЇНА"""
3,4,13674036,"ПРИВАТНЕ ПІДПРИЄМСТВО ""КОСІБА"""
4,5,13678525,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""СІРІУС"""
5,6,13680031,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""НАУКОВО-ТЕХНІЧНИЙ ЦЕНТР ""СКІФ"", ЛТД"""
6,7,14279974,"МАЛЕ ПІДПРИЄМСТВО ""НЦ - ПРИКЛАДНА МАТЕМАТИКА"""
7,8,14348132,"ПРИВАТНЕ ПІДПРИЄМСТВО ""СЕМЕНІВ"""
8,9,14355646,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ФОБОС-2"""
9,10,14360311,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""НІМЕЦЬКО-УКРАЇНСЬКА ФІРМА ""ДУАЦ"""


### 3.3. З'єднання (JOIN) за адресою реєстрації

Адреси, на яких зареєстровані і юридичні особи, і ФОП.

In [19]:
hive.sql(r"""
SELECT
    COUNT(DISTINCT u.address) AS shared_addresses,
    COUNT(*) AS pairs

FROM
    uo_table AS u

INNER JOIN
    fop_table AS f
    ON u.address = f.address
""")

[101.70s, HDFS read 3.0 GB] SELECT COUNT(DISTINCT u.address) AS shared_addresses, COUNT(*) AS pairs FROM uo_table AS u INNER ...


,shared_addresses,pairs
0,122917,1870748


In [20]:
hive.sql(r"""
SELECT
    u.address,
    COUNT(DISTINCT u.edrpou) AS uo_count,
    COUNT(DISTINCT f.fio) AS fop_count,
    COUNT(*) AS pairs

FROM
    uo_table AS u

INNER JOIN
    fop_table AS f
    ON u.address = f.address

GROUP BY u.address

ORDER BY pairs DESC

LIMIT 20
""")

[140.47s, HDFS read 3.3 GB] SELECT u.address, COUNT(DISTINCT u.edrpou) AS uo_count, COUNT(DISTINCT f.fio) AS fop_count, COUNT...


,address,uo_count,fop_count,pairs
0,"Україна, Кіровоградська обл., місто Кіровоград(п)",210,992,208530
1,"Україна, 61024, Харківська обл., місто Харків, ВУЛИЦЯ ПУШКІНСЬКА, будинок 79/1",60,361,22560
2,"Україна, 59052, Чернівецька обл., Сторожинецький р-н, село Великий Кучурів",23,835,20217
3,"Україна, 60014, Чернівецька обл., Хотинський р-н, село Клішківці",32,511,17728
4,"Україна, 79000, Львівська обл., місто Львів",98,162,15876
5,"Україна, 60321, Чернівецька обл., Новоселицький р-н, село Бояни",40,367,15600
6,"Україна, 59222, Чернівецька обл., Вижницький р-н, село Іспас",26,554,15366
7,"Україна, 60309, Чернівецька обл., Новоселицький р-н, село Маршинці",21,653,15183
8,"Україна, 60312, Чернівецька обл., Новоселицький р-н, село Рідківці",30,409,14070
9,"Україна, 60313, Чернівецька обл., Новоселицький р-н, село Магала",23,498,12420


## Завдання 4. Оптимізація зберігання даних (Partitioning & Bucketing)

Базова керована таблиця `uo_plain` — копія `uo_table` у тому ж форматі, що й оптимізовані таблиці (TEXTFILE), але без партицій і бакетів. Порівняння з нею показує саме ефект партиціювання/бакетингу, а не різницю між `OpenCSVSerde` і стандартним форматом.

In [21]:
hive.sql(r"""
DROP TABLE IF EXISTS uo_plain;

CREATE TABLE uo_plain
STORED AS TEXTFILE
AS
SELECT *
FROM uo_table
""")

[0.05s] DROP TABLE IF EXISTS uo_plain


[28.71s, HDFS read 1.1 GB] CREATE TABLE uo_plain STORED AS TEXTFILE AS SELECT * FROM uo_table


### 4.1. Партиціювання

Керована таблиця `uo_part`, партиціонована за станом юридичної особи (`stan`). Партиції створюються динамічно під час `INSERT INTO ... SELECT`.

In [22]:
hive.sql(r"""
DROP TABLE IF EXISTS uo_part;

CREATE TABLE uo_part (
    name     STRING,
    edrpou   STRING,
    address  STRING,
    boss     STRING,
    founders STRING,
    kved     STRING
)
PARTITIONED BY (stan STRING)
STORED AS TEXTFILE
""")

[0.03s] DROP TABLE IF EXISTS uo_part
[0.06s] CREATE TABLE uo_part ( name STRING, edrpou STRING, address STRING, boss STRING, founders STRING, ...


In [23]:
hive.sql(r"""
SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

INSERT INTO TABLE uo_part PARTITION (stan)
SELECT
    name,
    edrpou,
    address,
    boss,
    founders,
    kved,
    stan

FROM
    uo_table
""")

[0.01s] SET hive.exec.dynamic.partition=true
[0.00s] SET hive.exec.dynamic.partition.mode=nonstrict


[61.08s, HDFS read 1.2 GB] INSERT INTO TABLE uo_part PARTITION (stan) SELECT name, edrpou, address, boss, founders, kved, st...


Кожна партиція — окрема піддиректорія `stan=<значення>` у HDFS:

In [24]:
hdfs("dfs", "-ls", "-d", "/user/hive/warehouse/uo_part/stan=*")
hdfs("dfs", "-du", "-s", "-h", "/user/hive/warehouse/uo_part/stan=*");

$ hdfs dfs -ls -d '/user/hive/warehouse/uo_part/stan=*'
drwxr-xr-x   - hive supergroup          0 2026-10-07 22:06 /user/hive/warehouse/uo_part/stan=в стані припинення
drwxr-xr-x   - hive supergroup          0 2026-10-07 22:06 /user/hive/warehouse/uo_part/stan=зареєстровано
drwxr-xr-x   - hive supergroup          0 2026-10-07 22:06 /user/hive/warehouse/uo_part/stan=зареєстровано, свідоцтво про державну реєстрацію недійсне
drwxr-xr-x   - hive supergroup          0 2026-10-07 22:06 /user/hive/warehouse/uo_part/stan=порушено справу про банкрутство
drwxr-xr-x   - hive supergroup          0 2026-10-07 22:06 /user/hive/warehouse/uo_part/stan=порушено справу про банкрутство (санація)
drwxr-xr-x   - hive supergroup          0 2026-10-07 22:06 /user/hive/warehouse/uo_part/stan=припинено
drwxr-xr-x   - hive supergroup          0 2026-10-07 22:06 /user/hive/warehouse/uo_part/stan=скасовано


$ hdfs dfs -du -s -h '/user/hive/warehouse/uo_part/stan=*'
72.2 M   72.2 M   /user/hive/warehouse/uo_part/stan=в стані припинення
773.4 M  773.4 M  /user/hive/warehouse/uo_part/stan=зареєстровано
25.7 M   25.7 M   /user/hive/warehouse/uo_part/stan=зареєстровано, свідоцтво про державну реєстрацію недійсне
1.0 M    1.0 M    /user/hive/warehouse/uo_part/stan=порушено справу про банкрутство
83.4 K   83.4 K   /user/hive/warehouse/uo_part/stan=порушено справу про банкрутство (санація)
186.9 M  186.9 M  /user/hive/warehouse/uo_part/stan=припинено
2.6 M    2.6 M    /user/hive/warehouse/uo_part/stan=скасовано


Запит з фільтром за ключем партиціювання: `uo_part` читає лише одну піддиректорію, інші таблиці — всі дані.

In [25]:
STAN_QUERY = r"""
SELECT
    COUNT(*) AS companies

FROM
    {table}

WHERE 1=1
    AND stan = 'порушено справу про банкрутство'
"""

compare({table: STAN_QUERY.format(table=table) for table in ["uo_table", "uo_plain", "uo_part"]})

median 30.08s (runs: 23.62, 31.84, 30.08), HDFS read 1.1 GB | SELECT COUNT(*) AS companies FROM uo_table WHERE 1=1 AND stan = 'порушено справу про банкрутство'


median 25.32s (runs: 25.07, 25.57, 25.32), HDFS read 1.1 GB | SELECT COUNT(*) AS companies FROM uo_plain WHERE 1=1 AND stan = 'порушено справу про банкрутство'


median 23.92s (runs: 24.03, 23.92, 23.48), HDFS read 1.0 MB | SELECT COUNT(*) AS companies FROM uo_part WHERE 1=1 AND stan = 'порушено справу про банкрутство'


,"median, s","HDFS read, MB"
uo_table,30.08,1144.6
uo_plain,25.32,1115.9
uo_part,23.92,1.0


### 4.2. Бакетинг

Таблиця `uo_bucket` з 10 бакетами за `edrpou`. У Hive 3 параметр `hive.enforce.bucketing` видалено (бакетинг завжди примусовий).

`bucketing_version = 1`: у Hive 3 за замовчуванням версія 2 (хеш Murmur3), але `TABLESAMPLE` на MapReduce вибирає рядки старою хеш-функцією і для таких таблиць повертає неправильний (порожній) результат.

In [26]:
hive.sql(r"""
DROP TABLE IF EXISTS uo_bucket;

CREATE TABLE uo_bucket (
    name     STRING,
    edrpou   STRING,
    address  STRING,
    boss     STRING,
    founders STRING,
    kved     STRING,
    stan     STRING
)
CLUSTERED BY (edrpou) INTO 10 BUCKETS
STORED AS TEXTFILE
TBLPROPERTIES ("bucketing_version" = "1")
""")

[0.02s] DROP TABLE IF EXISTS uo_bucket
[0.05s] CREATE TABLE uo_bucket ( name STRING, edrpou STRING, address STRING, boss STRING, founders STRING...


In [27]:
hive.sql(r"""
INSERT INTO TABLE uo_bucket
SELECT
    name,
    edrpou,
    address,
    boss,
    founders,
    kved,
    stan

FROM
    uo_table
""")

[75.37s, HDFS read 1.1 GB] INSERT INTO TABLE uo_bucket SELECT name, edrpou, address, boss, founders, kved, stan FROM uo_table


10 бакетів — 10 файлів приблизно однакового розміру:

In [28]:
hdfs("dfs", "-ls", "-h", "/user/hive/warehouse/uo_bucket");

$ hdfs dfs -ls -h /user/hive/warehouse/uo_bucket
Found 11 items
drwxr-xr-x   - hive supergroup          0 2026-10-07 22:12 /user/hive/warehouse/uo_bucket/.hive-staging_hive_2026-10-07_22-10-49_975_8567143884801938769-1
-rw-r--r--   1 hive supergroup    111.6 M 2026-10-07 22:11 /user/hive/warehouse/uo_bucket/000000_0
-rw-r--r--   1 hive supergroup    111.8 M 2026-10-07 22:11 /user/hive/warehouse/uo_bucket/000001_0
-rw-r--r--   1 hive supergroup    112.5 M 2026-10-07 22:11 /user/hive/warehouse/uo_bucket/000002_0
-rw-r--r--   1 hive supergroup    110.9 M 2026-10-07 22:11 /user/hive/warehouse/uo_bucket/000003_0
-rw-r--r--   1 hive supergroup    111.0 M 2026-10-07 22:11 /user/hive/warehouse/uo_bucket/000004_0
-rw-r--r--   1 hive supergroup    113.2 M 2026-10-07 22:11 /user/hive/warehouse/uo_bucket/000005_0
-rw-r--r--   1 hive supergroup    112.0 M 2026-10-07 22:11 /user/hive/warehouse/uo_bucket/000006_0
-rw-r--r--   1 hive supergroup    110.0 M 2026-10-07 22:11 /user/hive/warehouse/uo_bucke

Точковий запит за `edrpou` до всіх трьох таблиць:

In [29]:
EDRPOU = "00223941"

POINT_QUERY = r"""
SELECT
    name,
    edrpou,
    stan

FROM
    {table}

WHERE 1=1
    AND edrpou = '{edrpou}'
"""

hive.sql(POINT_QUERY.format(table="uo_bucket", edrpou=EDRPOU))

[19.16s, HDFS read 1.1 GB] SELECT name, edrpou, stan FROM uo_bucket WHERE 1=1 AND edrpou = '00223941'


,name,edrpou,stan
0,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДАШУКІВСЬКІ БЕНТОНІТИ""",00223941,зареєстровано


In [30]:
point_timings = compare(
    {table: POINT_QUERY.format(table=table, edrpou=EDRPOU) for table in ["uo_table", "uo_plain", "uo_bucket"]}
)
point_timings

median 26.95s (runs: 26.95, 27.64, 26.42), HDFS read 1.1 GB | SELECT name, edrpou, stan FROM uo_table WHERE 1=1 AND edrpou = '00223941'


median 22.90s (runs: 22.91, 22.85, 22.90), HDFS read 1.1 GB | SELECT name, edrpou, stan FROM uo_plain WHERE 1=1 AND edrpou = '00223941'


median 22.05s (runs: 22.05, 22.26, 21.83), HDFS read 1.1 GB | SELECT name, edrpou, stan FROM uo_bucket WHERE 1=1 AND edrpou = '00223941'


,"median, s","HDFS read, MB"
uo_table,26.95,1144.5
uo_plain,22.90,1115.9
uo_bucket,22.05,1115.9


Hive на MapReduce не відсікає бакети за умовою `WHERE` (bucket pruning є лише в Tez), тому точковий запит все одно читає всі 10 файлів. Віртуальна колонка `INPUT__FILE__NAME` показує, у якому бакеті лежить запис:

In [31]:
bucket_file = hive.sql(rf"""
SELECT
    INPUT__FILE__NAME AS file,
    edrpou

FROM
    uo_bucket

WHERE 1=1
    AND edrpou = '{EDRPOU}'
""")
bucket_file

[22.01s, HDFS read 1.1 GB] SELECT INPUT__FILE__NAME AS file, edrpou FROM uo_bucket WHERE 1=1 AND edrpou = '00223941'


,file,edrpou
0,hdfs://namenode:9000/user/hive/warehouse/uo_bucket/000005_0,00223941


TABLESAMPLE сам обчислює хеш edrpou і повертає лише рядки бакета 6. Запис повернуто, тобто обчислений бакет збігся з файлом, у який Hive записав його при вставці (000005_0). Проте Hive 3.1.3 на MapReduce не пропускає файли інших бакетів: він читає всі дані й відбирає рядки за хешем (див. HDFS read).

In [32]:
bucket = int(Path(bucket_file.loc[0, "file"]).name.split("_")[0]) + 1

SAMPLE_QUERY = rf"""
SELECT
    name,
    edrpou,
    stan

FROM
    uo_bucket TABLESAMPLE(BUCKET {bucket} OUT OF 10 ON edrpou)

WHERE 1=1
    AND edrpou = '{EDRPOU}'
"""

display(hive.sql(SAMPLE_QUERY))
pd.concat([point_timings, compare({"uo_bucket TABLESAMPLE": SAMPLE_QUERY})])

[22.42s, HDFS read 1.1 GB] SELECT name, edrpou, stan FROM uo_bucket TABLESAMPLE(BUCKET 6 OUT OF 10 ON edrpou) WHERE 1=1 AND ...


,name,edrpou,stan
0,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДАШУКІВСЬКІ БЕНТОНІТИ""",00223941,зареєстровано


median 22.89s (runs: 21.21, 22.89, 23.45), HDFS read 1.1 GB | SELECT name, edrpou, stan FROM uo_bucket TABLESAMPLE(BUCKET 6 OUT OF 10 ON edrpou) WHERE 1=1 AND ...


,"median, s","HDFS read, MB"
uo_table,26.95,1144.5
uo_plain,22.90,1115.9
uo_bucket,22.05,1115.9
uo_bucket TABLESAMPLE,22.89,1115.9


## Завдання 5. WordCount за допомогою вбудованої бібліотеки

### 5.1. Підготовка вхідних даних

Логи HDFS з реального Hadoop-кластера: набір `HDFS_v1` з колекції [Loghub](https://github.com/logpai/loghub) (CC BY 4.0), ≈ 1,5 ГБ, 11,2 млн рядків.

In [33]:
logs = Path("data/wordcount/HDFS.log")
if not logs.exists():
    logs.parent.mkdir(parents=True, exist_ok=True)
    archive = logs.with_name("HDFS_v1.zip")
    urllib.request.urlretrieve("https://zenodo.org/records/8196385/files/HDFS_v1.zip?download=1", archive)
    partial = logs.with_suffix(".part")
    with zipfile.ZipFile(archive) as zf, zf.open("HDFS.log") as src, partial.open("wb") as dst:
        shutil.copyfileobj(src, dst)
    partial.rename(logs)
    archive.unlink()

hdfs("dfs", "-rm", "-r", "-f", "/wordcount/input", echo=False)
hdfs("dfs", "-mkdir", "-p", "/wordcount/input")
hdfs("dfs", "-put", "-f", "/data/wordcount/HDFS.log", "/wordcount/input/")
hdfs("dfs", "-ls", "-h", "/wordcount/input");

$ hdfs dfs -mkdir -p /wordcount/input


$ hdfs dfs -put -f /data/wordcount/HDFS.log /wordcount/input/


$ hdfs dfs -ls -h /wordcount/input
Found 1 items
-rw-r--r--   1 root supergroup      1.5 G 2026-10-07 22:19 /wordcount/input/HDFS.log


### 5.2. Запуск MapReduce-задачі

Запускаємо `wordcount` з 1, 2 та 4 редюсерами. Лог показано для запуску з 2 редюсерами.

In [34]:
REDUCERS = [1, 2, 4]

for reducers in REDUCERS:
    output = f"/wordcount/output-r{reducers}"
    hdfs("dfs", "-rm", "-r", "-f", output, echo=False)
    hadoop(
        "jar", EXAMPLES_JAR, "wordcount",
        "-D", f"mapreduce.job.reduces={reducers}",
        "/wordcount/input", output,
        echo=reducers == 2,
        log_level="INFO" if reducers == 2 else "WARN",
    )

$ hadoop jar /opt/hadoop-3.1.3/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.1.3.jar wordcount -D mapreduce.job.reduces=2 /wordcount/input /wordcount/output-r2
2026-10-07 22:20:27,028 INFO client.RMProxy: Connecting to ResourceManager at resourcemanager/172.18.0.4:8032
2026-10-07 22:20:27,199 INFO client.AHSProxy: Connecting to Application History server at historyserver/172.18.0.6:10200
2026-10-07 22:20:27,380 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1791409673690_0049
2026-10-07 22:20:27,591 INFO input.FileInputFormat: Total input files to process : 1
2026-10-07 22:20:27,655 INFO mapreduce.JobSubmitter: number of splits:12
2026-10-07 22:20:27,785 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1791409673690_0049
2026-10-07 22:20:27,785 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-07 22:20:27,955 INFO conf.Configuration: resource-types.xml not found
2026-10-07 22:20:27,956 INFO 

### 5.3. Аналіз результатів

In [35]:
for reducers in REDUCERS:
    hdfs("dfs", "-ls", f"/wordcount/output-r{reducers}")

$ hdfs dfs -ls /wordcount/output-r1
Found 2 items
-rw-r--r--   1 root supergroup          0 2026-10-07 22:20 /wordcount/output-r1/_SUCCESS
-rw-r--r--   1 root supergroup  140118458 2026-10-07 22:20 /wordcount/output-r1/part-r-00000


$ hdfs dfs -ls /wordcount/output-r2
Found 3 items
-rw-r--r--   1 root supergroup          0 2026-10-07 22:21 /wordcount/output-r2/_SUCCESS
-rw-r--r--   1 root supergroup   69939325 2026-10-07 22:21 /wordcount/output-r2/part-r-00000
-rw-r--r--   1 root supergroup   70179133 2026-10-07 22:21 /wordcount/output-r2/part-r-00001


$ hdfs dfs -ls /wordcount/output-r4
Found 5 items
-rw-r--r--   1 root supergroup          0 2026-10-07 22:22 /wordcount/output-r4/_SUCCESS
-rw-r--r--   1 root supergroup   35030823 2026-10-07 22:22 /wordcount/output-r4/part-r-00000
-rw-r--r--   1 root supergroup   34645449 2026-10-07 22:22 /wordcount/output-r4/part-r-00001
-rw-r--r--   1 root supergroup   34908502 2026-10-07 22:22 /wordcount/output-r4/part-r-00002
-rw-r--r--   1 root supergroup   35533684 2026-10-07 22:22 /wordcount/output-r4/part-r-00003


In [36]:
def read_wordcount(output: str) -> pd.DataFrame:
    listing = hdfs("dfs", "-ls", output, echo=False)
    rows = []
    for part in sorted(line.split()[-1] for line in listing.splitlines() if "/part-r-" in line):
        for line in hdfs("dfs", "-cat", part, echo=False).splitlines():
            word, count = line.rsplit("\t", 1)
            rows.append((Path(part).name, word, int(count)))
    return pd.DataFrame(rows, columns=["part", "word", "count"])


results = {reducers: read_wordcount(f"/wordcount/output-r{reducers}") for reducers in REDUCERS}

distribution = pd.concat(
    {
        reducers: df.groupby("part").agg(words=("word", "size"), occurrences=("count", "sum"))
        for reducers, df in results.items()
    },
    names=["reducers", "part"],
)
distribution["words, %"] = (
    distribution["words"] / distribution.groupby(level="reducers")["words"].transform("sum") * 100
).round(1)
distribution

words  occurrences  words, %
reducers part                                        
1        part-r-00000  3585666    138554843     100.0
2        part-r-00000  1791230     68658022      50.0
         part-r-00001  1794436     69896821      50.0
4        part-r-00000   895736     38982586      25.0
         part-r-00001   892027     30076772      24.9
         part-r-00002   895494     29675436      25.0
         part-r-00003   902409     39820049      25.2

Кожне слово потрапляє рівно в один файл, а загальні підсумки не залежать від кількості редюсерів:

In [37]:
pd.DataFrame(
    {
        reducers: {
            "distinct words": df["word"].nunique(),
            "words in >1 part": df["word"].duplicated().sum(),
            "total occurrences": df["count"].sum(),
        }
        for reducers, df in results.items()
    }
).T.rename_axis("reducers")

,distinct words,words in >1 part,total occurrences
reducers,,,
1,3585666,0,138554843
2,3585666,0,138554843
4,3585666,0,138554843


Початок кожного файлу (2 редюсери): всередині файлу слова відсортовані, але між файлами розподілені за хешем:

In [38]:
results[2].groupby("part").head(8)

,part,word,count
0,part-r-00000,/10.250.10.100,10393
1,part-r-00000,/10.250.10.100:,1755
2,part-r-00000,/10.250.10.100:32771,2
3,part-r-00000,/10.250.10.100:32780,1
4,part-r-00000,/10.250.10.100:32784,1
5,part-r-00000,/10.250.10.100:32786,1
6,part-r-00000,/10.250.10.100:32810,2
7,part-r-00000,/10.250.10.100:32816,1
1791230,part-r-00001,/10.250.10.100:32776,2
1791231,part-r-00001,/10.250.10.100:32792,1


Найчастіші слова:

In [39]:
results[2].nlargest(15, "count").reset_index(drop=True)

,part,word,count
0,part-r-00001,INFO,10812836
1,part-r-00000,block,6995672
2,part-r-00001,081111,5362327
3,part-r-00000,081110,4934281
4,part-r-00000,to,3942855
5,part-r-00000,BLOCK*,3700245
6,part-r-00001,dfs.FSNamesystem:,3700245
7,part-r-00000,size,3435615
8,part-r-00000,dfs.DataNode$PacketResponder:,3413350
9,part-r-00001,67108864,3227002


## Висновки

- **HDFS.** Файли зберігаються блоками по 128 МБ (UO.csv ≈ 1,2 ГБ → 9 блоків) з фактором реплікації 1, бо в кластері один DataNode.
- **Hive vs PostgreSQL.** `count(*)` у Hive триває ≈ 30–40 с, у PostgreSQL — ≈ 0,2–0,3 с. На ≈ 3 ГБ даних, що вміщуються на один сервер, домінують накладні витрати MapReduce (запуск задачі в YARN, JVM контейнерів). Hive виправданий на обсягах, які не вміщуються на одну машину.
- **Віконна функція.** `row_number()` за адресою виявляє адреси масової реєстрації: найбільша — Київ, вул. Мельникова, 12 (2187 юридичних осіб).
- **JOIN.** 122 917 адрес спільні для юридичних осіб і ФОП (1,87 млн пар); найбільше пар дають неповні адреси на кшталт «місто Кіровоград(п)».
- **Партиціювання.** Запит з фільтром за `stan` до `uo_part` читає лише одну партицію (≈ 1 МБ замість ≈ 1,1 ГБ). Час скорочується незначно, бо ≈ 20 с — фіксовані витрати на запуск MapReduce-задачі; на більших даних виграш пропорційний обсягу пропущених партицій. `uo_table` повільніша за `uo_plain` через розбір CSV (`OpenCSVSerde`).
- **Бакетинг.** Дані рівномірно розподілено на 10 файлів по ≈ 111 МБ, але точковий запит не прискорюється: на MapReduce Hive не відсікає бакети ні за `WHERE`, ні в `TABLESAMPLE`. Переваги бакетингу — bucket map join / sort-merge-bucket join і bucket pruning у рушії Tez.
- **WordCount.** На ≈ 1,5 ГБ логів HDFS (11,2 млн рядків, 138,6 млн слів) задача запускає 12 map-задач — по одній на блок HDFS. Combiner агрегує пари (слово, 1) ще на стороні map: до редюсерів надходить 4,4 млн пар замість 138,6 млн. Слова розподіляються між редюсерами за хешем ключа (`HashPartitioner`): кожне з 3,59 млн різних слів потрапляє рівно в один файл, файли майже рівні за кількістю різних слів (50/50, 25 % × 4), але не за кількістю входжень — часті слова (`INFO` — 10,8 млн) зосереджують багато входжень в одному файлі (від 29,7 до 39,8 млн при 4 редюсерах). Підсумки не залежать від кількості редюсерів.